
***

## 🧭 Part 1: Types of Time-related Interview Questions

### 1) **Timestamp Parsing & Normalization**

*   Convert strings to timestamps.
*   Handle time zones (e.g., stored in UTC, reported in IST).
*   Round/floor/ceil to date or hour buckets.
*   Deal with missing/invalid timestamps.

### 2) **Time-based Aggregations**

*   Sum/count per **day/week/month/hour**.
*   Group by local date after TZ conversion.
*   Calculate **day-of-week** patterns, weekend/holiday flags.

### 3) **Time Differences & Durations**

*   Compute user **session duration**, **shift hours**, **SLA elapsed time**.
*   IN→OUT pairing using **LEAD/LAG**.
*   Handle missing OUT/IN, overlapping intervals.

### 4) **Rolling & Cumulative Metrics**

*   Rolling **N-day averages/medians**.
*   Cumulative totals (running sum) by date per user.
*   Moving windows aligned to calendar boundaries.

### 5) **Sessionization (Gaps & Islands)**

*   Create sessions when **gap > threshold** (e.g., 30 minutes).
*   Identify session start/end, duration, event counts.

### 6) **Event Ordering & De-duplication**

*   Sort by timestamp, resolve duplicates/late arrivals.
*   Choose **first/last** event per day/user.

### 7) **Interval Logic (Overlaps & Joins)**

*   Detect overlapping shifts/bookings.
*   Interval joins (events falling inside availability windows).
*   Split across midnight and attribute properly.

### 8) **Calendar & Business Logic**

*   Workdays vs weekends.
*   Late/early arrival flags.
*   Custom day boundary (e.g., warehouse day 6AM–5:59AM).

### 9) **Cohorts & Retention (Time-based)**

*   First-activity cohort (by month), retention on D+1, D+7.
*   Active users by calendar period vs rolling windows.

### 10) **Resampling & Time Series**

*   Fill missing dates/hours with zeros.
*   Resample to uniform frequency; interpolate if needed.

***

## ⚠️ Pitfalls & Best Practices

*   Always **store in UTC**, **report in local TZ** (IST for Chennai).
*   Use **window functions** (LEAD/LAG) to pair events; filter cleanly.
*   Decide business rules for **missing OUT** or **multiple INs**.
*   Be explicit about **rolling window frame** (rows vs range).
*   For cross-midnight shifts, **define attribution** (start day vs split).

***


In [0]:
%sql
-- DROP TABLE my_table

In [0]:
%sql
CREATE OR REPLACE TABLE my_table (
  id INT,
  expiry_date DATE
);

INSERT INTO my_table (id, expiry_date)
SELECT
  id,
  DATE_ADD('2025-12-09', id) AS expiry_date
FROM
  (SELECT posexplode(array_repeat(1, 30)) AS (id, _)) t;

In [0]:
%sql
SELECT * FROM my_table

In [0]:
%python
import pandas as pd
from datetime import date

df = spark.table("workspace.default.my_table").toPandas()
month_end = (pd.Timestamp.now() + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')
month_end = pd.to_datetime(month_end,format = '%Y-%m-%d')

df_filtered = df[df['expiry_date'] < pd.Timestamp(month_end).date()]
print(df_filtered)

current_date = pd.to_datetime(str(date.today()), format = '%Y-%m-%d')
print(current_date)
print(type(current_date))

cur_date = pd.to_datetime(pd.Timestamp.now().date(), format= '%Y-%m-%d')
print(cur_date)


In [0]:
%python
curr_date = pd.to_datetime('today').strftime('%Y-%m-%d')
month_end = (pd.to_datetime('today') + pd.offsets.MonthEnd(0)).date()
print(month_end)

#15 days from today
days_15_from_today = (pd.Timestamp.now() + pd.Timedelta(days=15)).date()
print(days_15_from_today)


In [0]:
%python
import pandas as pd
from datetime import *
from calendar import monthrange
cards = [
    {"number": "1234", "expiry": "12/2025"},
    {"number": "5678", "expiry": "12/2025"},
    {"number": "9101", "expiry": "01/2026"}
]

today = date.today().strftime('%m/%Y')
print('current_mont&year: ', today)
filtered_card = []
for dic in cards:
    if dic['expiry'] == today:
        filtered_card.append(dic['number'])

filt_lst = [card['number'] for card in cards if card['expiry'] == today]
print('cared_before_expiry', filt_lst)

cards = [
    {"number": "1234", "expiry": "01/12/2025"},
    {"number": "5678", "expiry": "15/12/2025"},
    {"number": "9101", "expiry": "02/01/2026"}
]

curr_date = date.today()
print(monthrange(curr_date.year, curr_date.month))

month_end = date(
    curr_date.year, curr_date.month, monthrange(curr_date.year, curr_date.month)[1]
).strftime('%d/%m/%Y')
print('Month end date:', month_end)
curr_date = curr_date.strftime('%d/%m/%Y')

cardbeforexpiry = [
    card for card in cards
    if datetime.strptime(card['expiry'], '%d/%m/%Y') >= datetime.strptime( curr_date, '%d/%m/%Y') and
       datetime.strptime(card['expiry'], '%d/%m/%Y') <= datetime.strptime(month_end, '%d/%m/%Y')
]
print(cardbeforexpiry)

In [0]:

import pandas as pd
orders = pd.DataFrame([
    (1, 101, 2, 100),
    (2, 102, 1, 200),
    (3, 101, 3, 150),
    (4, 103, 5, 50),
    (5, 102, 2, 300)
], columns=["order_id", "customer_id", "quantity", "price"])

customers = pd.DataFrame([
    (101, "Alice"),
    (102, "Bob"),
    (103, "Charlie")
], columns=["customer_id", "name"])

df_join = pd.merge(customers, orders, on="customer_id", how="inner")
df_join['order_price']  = df_join['quantity'] * df_join['price']
df_result = df_join.groupby(['customer_id','name'])\
    .agg({'order_price': 'sum'})\
    .rename(columns = {'order_price':'total_sales'})\
    .sort_values(by = ['total_sales'], ascending = [False])\
    .reset_index()
df_result.display()





4. You are reading a file and parsing JSON. Handle cases where the file does not exist or JSON is invalid.
import json

with open('file.json','r') as f:
  data = json.load(f)

In [0]:
try:
    with open('file.json', 'r') as f:
        data = json.load(f)
except FileNotFoundError:
    print('File not found')
except json.JSONDecodeError:
    print('not valid json')
else:
    print(data)
finally:
    print('executed')


In [0]:
# . Upload a local file report.csv to an S3 bucket named my-data-bucket under the folder reports/.
%pip install boto3
import boto3


import boto3

s3 = boto3.client(
    's3',
    aws_access_key_id='your_access_key',
    aws_secret_access_key='your_secret_key',
    region_name='us-east-1'
)

bucket_name =  my-data-bucket
key = reports/report.csv
local_file = report.csv
s3.uploadfile(Filename = local_file, Key = key,Bucket = bucket_name )

In [0]:
import requests

api_url = "https://api.weatherapi.com/v1/current.json"
params = {
    "key": "YOUR_API_KEY",
    "q": "Chennai"
}

try:
    response = requests.get(api_url, params=params)
    response.raise_for_status()
    weather_data = response.json()
except requests.exceptions.HTTPError as e:
    print(f"HTTP error: {e}")
except ValueError:
    print("Invalid JSON response")
else:
    print(weather_data)
finally:
    print("Weather API fetch attempted")